In [0]:
%sql
-- Create target table (existing customers)
CREATE OR REPLACE TABLE customers (
    id        INT,
    name      STRING,
    email     STRING,
    status    STRING,
    balance   DOUBLE
);

-- Insert existing data
INSERT INTO customers VALUES
(1, 'Alice',   'alice@email.com',   'active',   5000.00),
(2, 'Bob',     'bob@email.com',     'active',   3000.00),
(3, 'Charlie', 'charlie@email.com', 'active',   7500.00),
(4, 'Diana',   'diana@email.com',   'inactive', 1200.00);

SELECT * FROM customers;

In [0]:
%sql
-- Source data arriving today with changes:
-- Customer 1 (Alice)   → balance updated
-- Customer 2 (Bob)     → status changed to inactive
-- Customer 5 (Eve)     → brand new customer
-- Customer 4 (Diana)   → not in source = no change

CREATE OR REPLACE TEMP VIEW incoming_updates AS
SELECT * FROM (
    VALUES
    (1, 'Alice',   'alice@email.com',   'active',   6500.00),
    (2, 'Bob',     'bob@email.com',     'inactive', 3000.00),
    (5, 'Eve',     'eve@email.com',     'active',   4500.00)
) AS t(id, name, email, status, balance);

SELECT * FROM incoming_updates;

In [0]:
%sql
MERGE INTO customers AS target
USING incoming_updates AS source
ON target.id = source.id

WHEN MATCHED AND source.balance != target.balance
    THEN UPDATE SET target.balance = source.balance

WHEN MATCHED AND source.status != target.status
    THEN UPDATE SET target.status = source.status

WHEN NOT MATCHED
    THEN INSERT (id, name, email, status, balance)
    VALUES (source.id, source.name, source.email, 
            source.status, source.balance);

-- Check results
SELECT * FROM customers ORDER BY id;

In [0]:
%sql
-- New source with a closed account flag
CREATE OR REPLACE TEMP VIEW closing_accounts AS
SELECT * FROM (
    VALUES
    (4, 'Diana', 'diana@email.com', 'closed', 0.00)
) AS t(id, name, email, status, balance);

-- MERGE with DELETE
MERGE INTO customers AS target
USING closing_accounts AS source
ON target.id = source.id

WHEN MATCHED AND source.status = 'closed'
    THEN DELETE

WHEN NOT MATCHED
    THEN INSERT (id, name, email, status, balance)
    VALUES (source.id, source.name, source.email,
            source.status, source.balance);

-- Check results
SELECT * FROM customers ORDER BY id;

In [0]:
%sql
-- Upsert = UPDATE if exists, INSERT if new
-- Most common MERGE pattern in production!

MERGE INTO customers AS target
USING incoming_updates AS source
ON target.id = source.id

WHEN MATCHED THEN
    UPDATE SET *    -- update ALL columns at once!

WHEN NOT MATCHED THEN
    INSERT *        -- insert ALL columns at once!

In [0]:
%sql

select * from customers;

In [0]:
%sql
select * from incoming_updates;

In [0]:
%sql
-- SCD Type 2 table — tracks full history
CREATE OR REPLACE TABLE customers_history (
    id           INT,
    name         STRING,
    email        STRING,
    status       STRING,
    balance      DOUBLE,
    is_current   BOOLEAN,
    valid_from   DATE,
    valid_to     DATE
);

-- Insert initial data with history columns
INSERT INTO customers_history VALUES
(1, 'Alice',   'alice@email.com',   'active', 5000.00, true,  '2026-01-01', null),
(2, 'Bob',     'bob@email.com',     'active', 3000.00, true,  '2026-01-01', null),
(3, 'Charlie', 'charlie@email.com', 'active', 7500.00, true,  '2026-01-01', null);

SELECT * FROM customers_history;

In [0]:
%sql
-- Step 1 — Expire the old record (set is_current = false)
UPDATE customers_history
SET 
    is_current = false,
    valid_to   = '2026-05-18'
WHERE id = 1 
AND is_current = true;

-- Step 2 — Insert new current record
INSERT INTO customers_history VALUES
(1, 'Alice', 'alice@email.com', 'active', 6500.00, true, '2026-05-18', null);

-- Check full history
SELECT * FROM customers_history 
WHERE id = 1
ORDER BY valid_from;

select * from customers_history

In [0]:
%sql
-- Bob's status changes to inactive
-- Using MERGE for SCD Type 2

MERGE INTO customers_history AS target
USING (
    SELECT 
        2          AS id,
        'Bob'      AS name,
        'bob@email.com' AS email,
        'inactive' AS status,
        3000.00    AS balance,
        '2026-05-18' AS change_date
) AS source
ON target.id = source.id 
AND target.is_current = true

WHEN MATCHED THEN
    UPDATE SET
        target.is_current = false,
        target.valid_to   = source.change_date

WHEN NOT MATCHED THEN
    INSERT (id, name, email, status, balance, 
            is_current, valid_from, valid_to)
    VALUES (source.id, source.name, source.email,
            source.status, source.balance,
            true, source.change_date, null);

-- Check Bob's history
SELECT * FROM customers_history
WHERE id = 2
ORDER BY valid_from;


-- Check Bob's current state first
SELECT * FROM customers_history
WHERE id = 2
ORDER BY valid_from;



DESCRIBE HISTORY customers_history